# 18 · Events, streaming and cancellation

Inspect the event stream directly. StreamingMode.SSE delivers text as it is generated; partial chunks and final aggregated content must not both be appended to the same display.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 1 streamed Lite call. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import RunConfig
from google.adk.agents.run_config import StreamingMode
agent = llm("streaming_shop", "Give three short tips for checking an invoice.")
lab = await make_lab(agent)
events = []
saw_partial = False
async for event in lab.runner.run_async(
    user_id=lab.user_id, session_id=lab.session.id,
    new_message=types.Content(role="user", parts=[types.Part(text="How do I check my invoice?")]),
    run_config=RunConfig(streaming_mode=StreamingMode.SSE, max_llm_calls=2)):
    events.append(event)
    text = "".join(p.text or "" for p in event.content.parts or []) if event.content else ""
    if event.partial:
        saw_partial = True
        print(text, end="", flush=True)
    elif event.is_final_response() and not saw_partial:
        print(text, end="")
print("\nReported tokens:", token_usage(events))


In [ ]:
import asyncio
# For UI cancellation, retain the task and call task.cancel(). The coroutine
# consuming run_async must use try/finally and close the generator/resources.
async def bounded_question(lab, question, seconds=30):
    async with asyncio.timeout(seconds):
        return await ask(lab, question)
print("bounded_question is ready; no additional model call was made.")


## Try it

Print only event author, partial and turn_complete fields to understand stream boundaries. Test cancellation with a short timeout in a separate run.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
